In [ ]:
import os
import rasterio
import tensorflow as tf
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from src.config import *
from tensorflow.keras import Sequential
from tensorflow.keras import layers,models
import matplotlib.pyplot as plt
import laspy
import numpy as np
from pathlib import Path

In [ ]:
input_data_dtm = ''
output_data_dtm = '/home/ajai-krishna/work/GEO_AI/data/Training/dtm/gujarath_dtm_RGB.tif'
patch_output_dir = '/home/ajai-krishna/work/GEO_AI/data/Training/Patches/'

In [ ]:
with rasterio.open(data_dtm) as src:

    # ── Basic Info ──────────────────────────────────────────
    print(f"Number of Bands : {src.count}")
    print(f"Width x Height  : {src.width} x {src.height}")
    print(f"Data Type       : {src.dtypes}")
    print(f"CRS             : {src.crs}")
    print(f"Transform       : {src.transform}")
    print(f"Nodata Value    : {src.nodata}")
    print(f"Driver          : {src.driver}")

    # ── Per-Band Stats ──────────────────────────────────────
    for i in range(1, src.count + 1):
        band = src.read(i).astype(np.float32)
        print(f"\n  Band {i}:")
        print(f"    Min    : {band.min():.4f}")
        print(f"    Max    : {band.max():.4f}")
        print(f"    Mean   : {band.mean():.4f}")
        print(f"    Std    : {band.std():.4f}")
        print(f"    Shape  : {band.shape}")
        print(f"    Dtype  : {band.dtype}")

        # Check if band has nodata / empty regions
        nodata_count = np.sum(band == src.nodata) if src.nodata else 0
        print(f"    Nodata pixels: {nodata_count}")

In [ ]:
def extract_patches(data_path, patch_size=256, stride=128, nodata_threshold=0.1, output_dir=None):

    # patch_output_dir.os.mkdir(parents=True, exist_ok=True)

    with rasterio.open(data_path) as src:
        data = src.read().astype(np.float32)       # (4, H, W)
        data = np.transpose(data, (1, 2, 0))       # (H, W, 4)

    H, W, _ = data.shape
    patch_id = 0
    skipped  = 0

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):

            patch = data[y:y+patch_size, x:x+patch_size, :]

            nodata_ratio = (patch == -9999.0).mean()
            if nodata_ratio > nodata_threshold:
                skipped += 1
                continue

            np.save(output_dir / f"patch_{patch_id}.npy", patch)
            patch_id += 1

    print(f"Total patches  : {patch_id + skipped}")
    print(f"Saved patches  : {patch_id}")
    print(f"Skipped patches: {skipped}  (nodata > {nodata_threshold*100}%)")
    return patch_id

In [ ]:
extract_patches(data_dtm, patch_size=256, stride=128, nodata_threshold=0.1, output_dir=Path(patch_output_dir))

In [ ]:
def normalize_patch(patch):
    """
    patch shape: (256, 256, 4)
    Band 0,1,2 → RGB  (16-bit float32, range 0–65535) → divide by 65535
    Band 3     → DTM  (elevation in metres)            → z-score
    """
    normalized = np.zeros_like(patch, dtype=np.float32)

    # ── RGB Bands (0, 1, 2) ───────────────────────────────
    rgb = patch[:, :, :3]
    rgb = np.clip(rgb, 0, 65535)
    normalized[:, :, :3] = rgb / 65535.0

    # ── DTM Band (3) → z-score, ignore nodata ─────────────
    dtm   = patch[:, :, 3]
    valid = dtm != -9999.0
    if valid.sum() > 0:
        mean = dtm[valid].mean()
        std  = dtm[valid].std() + 1e-8
        normalized[:, :, 3] = np.where(valid, (dtm - mean) / std, 0.0)

    return normalized


# ── Normalize and save back to disk (avoid RAM overload) ──
patches = sorted([f for f in os.listdir(patch_output_dir) if f.endswith('.npy')])

norm_output_dir = Path('/home/ajai-krishna/work/GEO_AI/data/Training/Patches_Normalized/')
norm_output_dir.mkdir(parents=True, exist_ok=True)

for i, patch_file in enumerate(patches):
    patch      = np.load(os.path.join(patch_output_dir, patch_file))  # (256, 256, 4)
    normalized = normalize_patch(patch)
    np.save(norm_output_dir / patch_file, normalized)                  # save same filename

    if i % 500 == 0:
        print(f"Normalized {i}/{len(patches)} patches")

print(f"Done — {len(patches)} patches normalized → {norm_output_dir}")

In [ ]:
def augment(patch, label):
    """Spatial augmentation — same transform applied to both patch and label."""

    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        patch = tf.image.flip_left_right(patch)
        label = tf.image.flip_left_right(label[..., tf.newaxis])[..., 0]

    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        patch = tf.image.flip_up_down(patch)
        label = tf.image.flip_up_down(label[..., tf.newaxis])[..., 0]

    # Random brightness on RGB only (leave DTM band untouched)
    rgb   = tf.image.random_brightness(patch[:, :, :3], max_delta=0.1)
    patch = tf.concat([rgb, patch[:, :, 3:]], axis=-1)

    return patch, label


In [ ]:

def create_patches(image, patch_size=256):
    patches = []
    h, w = image.shape
    
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):

            patch = image[i:i+patch_size, j:j+patch_size]

            if patch.shape == (patch_size, patch_size):
                patches.append(patch)

    return np.array(patches)

In [ ]:
patches = create_patches(img, 256)

print(patches.shape)

In [ ]:
patches = np.expand_dims(patches, axis=-1)

print(patches.shape)

In [ ]:
patches = patches.astype("float32")
patches = (patches - patches.min()) / (patches.max() - patches.min())

In [ ]:
labels = np.random.rand(patches.shape[0], 1)

In [ ]:
labels.shape

In [ ]:
model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', padding='same', input_shape=(256,256,1)),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax'),
    layers.Dense(1, activation='sigmoid')# 2 classes
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    patches,
    labels,
    batch_size=16,
    epochs=10,
    validation_split=0.2
)

In [ ]:

# model = models.Sequential()

# model.add(layers.Conv2D(
#     filters=16,
#     kernel_size=(3,3),
#     activation='relu',
#     input_shape=(32,32,1)
# ))

# model.add(layers.MaxPooling2D(pool_size=(2,2)))

# model.add(layers.GlobalAveragePooling2D())

# model.add(layers.Dense(20, activation='softmax'))

# model.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

In [ ]:
model.fit(
    patches,
    labels,
    batch_size=16,
    epochs=10,
    validation_split=0.2
)